In [3]:
import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control

# ==========================================
# Input and output file names
# ==========================================
input_file = "Raw-p-val.xlsx"
output_file = "Raw-p-val(3)_FDR_corrected.xlsx"

# ==========================================
# Sheet names to process
# ==========================================
target_sheets = [
    "degree_centrality",
    "degree_centrality2",
    "betweenness_centrality",
    "betweenness_centrality2"
]

# ==========================================
# Read workbook
# ==========================================
xls = pd.ExcelFile(input_file)

# Store processed sheets here
processed_sheets = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(input_file, sheet_name=sheet)
    df = df.copy()

    # Only process the 4 target sheets
    if sheet in target_sheets:
        if "p-value" not in df.columns:
            print(f"[SKIP] Sheet '{sheet}' does not contain a 'p-value' column.")
            processed_sheets[sheet] = df
            continue

        # Convert p-values to numeric safely
        df["p-value"] = pd.to_numeric(df["p-value"], errors="coerce")

        # Mask for rows that actually have p-values
        mask = df["p-value"].notna()

        if mask.sum() == 0:
            print(f"[SKIP] Sheet '{sheet}' has no valid p-values.")
            df["FDR_BH_pvalue"] = np.nan
            df["FDR_BH_significant_0.05"] = np.nan
        else:
            # Benjamini-Hochberg correction within this sheet only
            corrected_pvals = false_discovery_control(
                df.loc[mask, "p-value"].to_numpy(),
                method="bh"
            )

            # Add new columns
            df["FDR_BH_pvalue"] = np.nan
            df["FDR_BH_significant_0.05"] = np.nan

            df.loc[mask, "FDR_BH_pvalue"] = corrected_pvals
            df.loc[mask, "FDR_BH_significant_0.05"] = corrected_pvals < 0.05

            print(f"[DONE] Sheet '{sheet}' corrected for {mask.sum()} p-values.")

    processed_sheets[sheet] = df

# ==========================================
# Save all sheets to a new Excel file
# ==========================================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet, df in processed_sheets.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

print(f"\nFinished. New file saved as: {output_file}")

[DONE] Sheet 'degree_centrality' corrected for 32 p-values.
[DONE] Sheet 'degree_centrality2' corrected for 4 p-values.
[DONE] Sheet 'betweenness_centrality' corrected for 22 p-values.
[DONE] Sheet 'betweenness_centrality2' corrected for 2 p-values.

Finished. New file saved as: Raw-p-val(3)_FDR_corrected.xlsx


/tmp/ipykernel_490/1566974108.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True False  True False  True  True  True  True  True False False False
  True  True  True False  True False  True False False False  True False
 False False False False False False False False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "FDR_BH_significant_0.05"] = corrected_pvals < 0.05
/tmp/ipykernel_490/1566974108.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True  True  True  True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "FDR_BH_significant_0.05"] = corrected_pvals < 0.05
/tmp/ipykernel_490/1566974108.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a